# 🧮 Optimización de Inventario - Modelo Newsvendor

## Problema
- Si compras **poco** → pierdes ventas (quiebre de stock)
- Si compras **mucho** → desperdicias (sobrestock)

## Pregunta clave
> **¿Cuánto debo comprar exactamente de cada insumo?**

## Enfoque técnico
1. **Forecast probabilístico** de demanda (distribución Normal)
2. **Modelo Newsvendor**: `Q* = μ_LT + z(SL) × σ_LT`
3. **Optimización bajo incertidumbre** con nivel de servicio configurable

## Resultado esperado
```
Compra óptima: 320 kg
Riesgo de faltante: 5%
Riesgo de desperdicio: 8%
```

---

### Pipeline de datos (dbt)
```
Silver (staging) → Intermediate (int_demanda_insumos)
                  → Features (feat_inventario_demanda)  
                  → Dataset (obt_inventory_optimization)
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import snowflake.connector
from dotenv import load_dotenv
import mlflow
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

load_dotenv()
print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


In [ ]:
# Conexión a Snowflake
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'),
    database=os.getenv('SNOWFLAKE_DATABASE', 'HGC_DB'),
    schema='TRAINING_DATASETS'
)

# Extraer dataset desde Snowflake (generado por dbt run)
query = 'SELECT * FROM TRAINING_DATASETS.OBT_INVENTORY_OPTIMIZATION'
df = pd.read_sql(query, conn)
df.columns = [c.upper() for c in df.columns]

print(f'✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

✅ Dataset cargado: 24 filas × 30 columnas


,ID_INSUMO_NK,ID_ALMACEN_NK,NOMBRE_INSUMO,UNIDAD_MEDIDA,FEATURE_DEMANDA_PROMEDIO_DIARIA,FEATURE_DEMANDA_DESVIACION_STD,FEATURE_COEFICIENTE_VARIACION,FEATURE_DEMANDA_MEDIANA_DIARIA,FEATURE_DEMANDA_MINIMA_DIARIA,FEATURE_DEMANDA_MAXIMA_DIARIA,...,FEATURE_COSTO_UNITARIO,FEATURE_PRECIO_COMPRA_PROMEDIO,FEATURE_LEAD_TIME_DIAS,FEATURE_DEMANDA_ESPERADA_LT,FEATURE_DESVIACION_DEMANDA_LT,FEATURE_NUM_PRODUCTOS_DEPENDIENTES,META_DIAS_CON_DEMANDA,META_PRIMERA_FECHA,META_ULTIMA_FECHA,META_RANGO_DIAS
0,1,4,Pollo Crudo Entero,Kg,176.2528,114.7671,0.6512,139.510,54.30,659.30,...,15.5,15.5,3.050578,537.6728,200.4511,3,3939,2016-03-20,2026-12-31,3938
1,1,22,Pollo Crudo Entero,Kg,88.4140,16.3386,0.1848,87.520,47.70,143.55,...,15.5,15.5,3.050578,269.7137,28.5369,3,1184,2023-10-05,2026-12-31,1183
2,1,12,Pollo Crudo Entero,Kg,116.1054,38.4863,0.3315,106.070,45.53,252.36,...,15.5,15.5,3.050578,354.1886,67.2199,3,2638,2019-10-12,2026-12-31,2637
3,1,20,Pollo Crudo Entero,Kg,94.4482,18.8575,0.1997,92.420,50.76,160.64,...,15.5,15.5,3.050578,288.1216,32.9363,3,1781,2022-02-15,2026-12-31,1780
4,6,14,Cajas de Cartón Combo,Unidad,96.9113,20.5966,0.2125,95.005,50.42,177.34,...,1.2,1.2,2.916053,282.5985,35.1717,7,1880,2021-11-08,2026-12-31,1879
